# Customer Health Score: Building a Churn Prediction Model
## Three Approaches to Predicting Churn from Adoption & Usage Data

> **Course:** Enterprise Architecture for SaaS Business Models  
> **Run in:** Google Colab - execute cells top to bottom.

---

## The Task

You have a **historical dataset** of SaaS customers where the outcome is already known — each customer either churned, renewed, or expanded. The signals in the dataset (seat utilisation, login frequency, features adopted, QBR recency, support tickets, NPS, training completion) were recorded *before* the outcome was determined.

**Your goal:** use this historical data to build a prediction model — a set of rules or scores that, given only the signals, can predict whether a customer will churn. Once validated on historical data, the same model is applied to *current* customers where the outcome is not yet known.

Each option below builds a different type of model, applies it back to the historical data, and measures how accurately it predicts the known outcomes.

| Approach | What it produces | Requires known outcomes? |
|---|---|---|
| **A - Weighted Score** | `Health_Score`, `Predicted_Churn` per customer | No — weights set manually |
| **B - Red Flag Counter** | `Red_Flag_Count`, `Predicted_Churn` per customer | No — thresholds set manually |
| **C - Signal Ranking** | `Churn_Risk_Score`, `Predicted_Churn` per customer | Yes — outcomes used for training |

---
## Setup - Load the Dataset

Run this cell first. All three option sections depend on it.

In [9]:
import pandas as pd, numpy as np

!wget -O HealthScore_500.xlsx 'https://github.com/btx4452/CS_HealthScore_Analysis/raw/main/HealthScore_500.xlsx'

df = pd.read_excel('./HealthScore_500.xlsx', sheet_name='Customer Data')
print(f"Loaded {len(df)} rows")
print("Columns:", list(df.columns))

display(df)

--2026-04-20 20:49:34--  https://github.com/btx4452/CS_HealthScore_Analysis/raw/main/HealthScore_500.xlsx
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/btx4452/CS_HealthScore_Analysis/main/HealthScore_500.xlsx [following]
--2026-04-20 20:49:35--  https://raw.githubusercontent.com/btx4452/CS_HealthScore_Analysis/main/HealthScore_500.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 59162 (58K) [application/octet-stream]
Saving to: ‘HealthScore_500.xlsx’

HealthScore_500.xls 100%[===================>]  57.78K  --.-KB/s    in 0.01s   

2026-04-20 20:49:35 (4.84 MB/s) - ‘HealthScore_500.xlsx’ s

,Customer_ID,Company_Name,Industry,ACV_USD,Seats_Licensed,Seats_Active_30d,Logins_Total_30d,Avg_Session_min,Features_Adopted,Reports_Generated_30d,Integrations_Active,Support_Tickets_Open,Support_Tickets_90d,Last_QBR_Days_Ago,NPS_Score,Payment_Overdue_Days,Training_Completion_pct,Outcome
0,O001,Spark Works SA,Retail,48000,70,43,721,20.9,3,11,2,2,8,71,55,0,56,Renewed
1,O002,Dynamic Synergies GmbH,Energy,55000,69,66,2194,47.6,4,116,4,0,2,43,39,0,80,Expanded
2,O003,Catalyst Works AB,Consulting,16000,23,5,17,6.3,1,10,0,4,5,252,-66,0,21,Churned
3,O004,Fusion Holdings AB,Technology,51000,76,63,3523,36.6,4,122,3,1,1,11,54,0,82,Expanded
4,O005,Titan Works Ltd,Retail,66000,86,70,2160,23.2,5,194,4,1,4,21,40,0,86,Expanded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,O496,Fusion Innovations Inc,Construction,64000,100,59,708,14.8,3,59,1,0,9,88,5,0,48,Renewed
496,O497,Cloud Capital Inc,MarTech,11000,16,4,27,3.7,1,5,0,5,5,355,-55,0,22,Churned
497,O498,Pinnacle Group AB,MarTech,78000,139,138,8213,38.9,5,126,5,0,7,46,37,0,77,Expanded
498,O499,Smart Innovations Corp,Media,65000,86,37,864,21.2,3,47,2,1,10,48,7,0,59,Renewed


---
## Option A - Weighted Health Score (0–100)

**How it works:** each signal is normalised to a 0–100 scale and multiplied by an assigned weight. The weighted values are summed to produce a single Health Score per customer.

| Signal | Column | Weight | Direction |
|---|---|---|---|
| Seat Utilisation Rate | `Seats_Active_30d / Seats_Licensed` | 25% | Higher = better |
| Login Frequency per Seat | `Logins_Total_30d / Seats_Active_30d` | 20% | Higher = better |
| Features Adopted | `Features_Adopted` | 20% | Higher = better |
| QBR Recency | `Last_QBR_Days_Ago` (inverted) | 15% | Lower = better |
| Open Support Tickets | `Support_Tickets_Open` (inverted) | 10% | Lower = better |
| NPS Score | `NPS_Score` | 5% | Higher = better |
| Training Completion | `Training_Completion_pct` | 5% | Higher = better |

**Score thresholds:** `< 40` = Predict Churn &nbsp;·&nbsp; `40–70` = Monitor &nbsp;·&nbsp; `> 70` = Healthy

**Added columns:** `sig_*` (normalised signals 0–100), `Health_Score`, `Predicted_Churn`, `Correct`

In [10]:
#@title Code Option A - Weighted Health Score
from google.colab import files
pd.options.display.float_format = '{:.1f}'.format

# ── Step 1: Normalise each signal to 0–100 ───────────────────────────────────
df['sig_seat_util'] = ((df['Seats_Active_30d'] / df['Seats_Licensed'].replace(0, 1)).clip(0, 1) * 100).round(1)
df['sig_logins']    = ((df['Logins_Total_30d']  / df['Seats_Active_30d'].replace(0, 1)).clip(0, 30) / 30 * 100).round(1)
df['sig_features']  = ((df['Features_Adopted']  / 5).clip(0, 1) * 100).round(1)
df['sig_qbr']       = ((1 - (df['Last_QBR_Days_Ago'] / 180).clip(0, 1)) * 100).round(1)
df['sig_tickets']   = ((1 - (df['Support_Tickets_Open'] / 10).clip(0, 1)) * 100).round(1)
df['sig_nps']       = (((df['NPS_Score'] + 100) / 200) * 100).round(1)
df['sig_training']  = df['Training_Completion_pct'].clip(0, 100).round(1)

# ── Step 2: Apply weights → Health Score ─────────────────────────────────────
# These weights are the MODEL SPECIFICATION — adjust and re-run to test variants
WEIGHTS = {
    'sig_seat_util': 0.25, 'sig_logins':   0.20, 'sig_features': 0.20,
    'sig_qbr':       0.15, 'sig_tickets':  0.10, 'sig_nps':      0.05,
    'sig_training':  0.05,
}
CHURN_THRESHOLD = 40   # Health Score below this → predict Churn

df['Health_Score']    = sum(df[col] * w for col, w in WEIGHTS.items()).round(1)
df['Predicted_Churn'] = (df['Health_Score'] < CHURN_THRESHOLD).map({True: 'Churn', False: 'No Churn'})
df['Actual_Churn']    = (df['Outcome'] == 'Churned').map({True: 'Churn', False: 'No Churn'})
df['Correct']         = (df['Predicted_Churn'] == df['Actual_Churn'])

# ── Step 3: Evaluate — how well does the model predict known outcomes? ────────
TP = ((df['Predicted_Churn']=='Churn') & (df['Actual_Churn']=='Churn')).sum()
FP = ((df['Predicted_Churn']=='Churn') & (df['Actual_Churn']=='No Churn')).sum()
FN = ((df['Predicted_Churn']=='No Churn') & (df['Actual_Churn']=='Churn')).sum()
TN = ((df['Predicted_Churn']=='No Churn') & (df['Actual_Churn']=='No Churn')).sum()

accuracy  = (TP + TN) / len(df)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

print(f"=== Model evaluation (threshold: Health Score < {CHURN_THRESHOLD}) ===")
print(f"Accuracy : {accuracy:.0%}  ({TP+TN}/{len(df)} correctly classified)")
print(f"Precision: {precision:.0%}  (of predicted churns, how many actually churned)")
print(f"Recall   : {recall:.0%}  (of actual churns, how many were caught)")
print(f"\nConfusion matrix:")
print(f"                 Predicted Churn  Predicted No Churn")
print(f"Actual Churn     {TP:>15}  {FN:>18}")
print(f"Actual No Churn  {FP:>15}  {TN:>18}")

print("\n=== Customers the model got WRONG ===")
display(df[df['Correct']==False][['Customer_ID','Company_Name','Outcome','Health_Score','Predicted_Churn']].reset_index(drop=True))

# ── Step 4: Full output ───────────────────────────────────────────────────────
KEY  = ['Customer_ID', 'Company_Name', 'Industry', 'ACV_USD']
CALC = ['Health_Score', 'Predicted_Churn', 'Correct',
        'sig_seat_util', 'sig_logins', 'sig_features',
        'sig_qbr', 'sig_tickets', 'sig_nps', 'sig_training']
RAW  = ['Seats_Licensed', 'Seats_Active_30d', 'Logins_Total_30d',
        'Features_Adopted', 'Last_QBR_Days_Ago', 'Support_Tickets_Open',
        'NPS_Score', 'Payment_Overdue_Days', 'Training_Completion_pct', 'Outcome']
print("\n=== All customers with calculated columns ===")
display(df[KEY + CALC + RAW].reset_index(drop=True))

=== Model evaluation (threshold: Health Score < 40) ===
Accuracy : 99%  (497/500 correctly classified)
Precision: 100%  (of predicted churns, how many actually churned)
Recall   : 98%  (of actual churns, how many were caught)

Confusion matrix:
                 Predicted Churn  Predicted No Churn
Actual Churn                 172                   3
Actual No Churn                0                 325

=== Customers the model got WRONG ===


,Customer_ID,Company_Name,Outcome,Health_Score,Predicted_Churn
0,O158,Swift Labs Corp,Churned,57.5,No Churn
1,O165,Flux Insights SA,Churned,42.4,No Churn
2,O190,Flux Innovations AB,Churned,51.4,No Churn



=== All customers with calculated columns ===


,Customer_ID,Company_Name,Industry,ACV_USD,Health_Score,Predicted_Churn,Correct,sig_seat_util,sig_logins,sig_features,...,Seats_Licensed,Seats_Active_30d,Logins_Total_30d,Features_Adopted,Last_QBR_Days_Ago,Support_Tickets_Open,NPS_Score,Payment_Overdue_Days,Training_Completion_pct,Outcome
0,O001,Spark Works SA,Retail,48000,62.3,No Churn,True,61.4,55.9,60.0,...,70,43,721,3,71,2,55,0,56,Renewed
1,O002,Dynamic Synergies GmbH,Energy,55000,88.8,No Churn,True,95.7,100.0,80.0,...,69,66,2194,4,43,0,39,0,80,Expanded
2,O003,Catalyst Works AB,Consulting,16000,19.6,Churn,True,21.7,11.3,20.0,...,23,5,17,1,252,4,-66,0,21,Churned
3,O004,Fusion Holdings AB,Technology,51000,87.8,No Churn,True,82.9,100.0,80.0,...,76,63,3523,4,11,1,54,0,82,Expanded
4,O005,Titan Works Ltd,Retail,66000,90.4,No Churn,True,81.4,100.0,100.0,...,86,70,2160,5,21,1,40,0,86,Expanded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,O496,Fusion Innovations Inc,Construction,64000,57.4,No Churn,True,59.0,40.0,60.0,...,100,59,708,3,88,0,5,0,48,Renewed
496,O497,Cloud Capital Inc,MarTech,11000,22.0,Churn,True,25.0,22.5,20.0,...,16,4,27,1,355,5,-55,0,22,Churned
497,O498,Pinnacle Group AB,MarTech,78000,93.3,No Churn,True,99.3,100.0,100.0,...,139,138,8213,5,46,0,37,0,77,Expanded
498,O499,Smart Innovations Corp,Media,65000,63.9,No Churn,True,43.0,77.8,60.0,...,86,37,864,3,48,1,7,0,59,Renewed


---
## Option B - Red Flag Counter

**How it works:** each signal is checked against a binary threshold. If the threshold is breached, one red flag is raised. The total count determines the risk tier.

| Signal | Column | Red Flag Condition |
|---|---|---|
| Seat Utilisation | `Seats_Active_30d / Seats_Licensed` | < 30% |
| Logins per Seat | `Logins_Total_30d / Seats_Active_30d` | < 5 per month |
| Features Adopted | `Features_Adopted` | ≤ 2 |
| Last QBR | `Last_QBR_Days_Ago` | > 90 days |
| Open Support Tickets | `Support_Tickets_Open` | ≥ 3 |
| NPS | `NPS_Score` | < 0 (Detractor) |
| Payment Overdue | `Payment_Overdue_Days` | > 0 |
| Training Completion | `Training_Completion_pct` | < 40% |

**Risk tiers:** `0–1` = Healthy &nbsp;·&nbsp; `2–3` = Watch &nbsp;·&nbsp; `4+` = Predict Churn

**Added columns:** `flag_*` (1 = flag raised, 0 = clear), `Red_Flag_Count`, `Predicted_Churn`, `Correct`

In [11]:
#@title Option B - Red Flag Counter
from google.colab import files
pd.options.display.float_format = '{:.1f}'.format

# ── Step 1: Apply thresholds → flags ─────────────────────────────────────────
# These thresholds are the MODEL SPECIFICATION — adjust and re-run to test variants
seat_util       = (df['Seats_Active_30d'] / df['Seats_Licensed'].replace(0, 1)).round(1)
logins_per_seat = (df['Logins_Total_30d']  / df['Seats_Active_30d'].replace(0, 1)).round(1)

df['flag_seat_util']  = (seat_util       < 0.30).astype(int)
df['flag_logins']     = (logins_per_seat < 5).astype(int)
df['flag_features']   = (df['Features_Adopted']        <= 2).astype(int)
df['flag_qbr']        = (df['Last_QBR_Days_Ago']       > 90).astype(int)
df['flag_tickets']    = (df['Support_Tickets_Open']    >= 3).astype(int)
df['flag_nps']        = (df['NPS_Score']               <  0).astype(int)
df['flag_payment']    = (df['Payment_Overdue_Days']    >  0).astype(int)
df['flag_training']   = (df['Training_Completion_pct'] < 40).astype(int)

FLAG_COLS   = [c for c in df.columns if c.startswith('flag_')]
CHURN_FLAGS = 4   # flag count at or above this → predict Churn

# ── Step 2: Count flags → prediction ─────────────────────────────────────────
df['Red_Flag_Count']  = df[FLAG_COLS].sum(axis=1)
df['Predicted_Churn'] = (df['Red_Flag_Count'] >= CHURN_FLAGS).map({True: 'Churn', False: 'No Churn'})
df['Actual_Churn']    = (df['Outcome'] == 'Churned').map({True: 'Churn', False: 'No Churn'})
df['Correct']         = (df['Predicted_Churn'] == df['Actual_Churn'])

# ── Step 3: Evaluate ──────────────────────────────────────────────────────────
TP = ((df['Predicted_Churn']=='Churn') & (df['Actual_Churn']=='Churn')).sum()
FP = ((df['Predicted_Churn']=='Churn') & (df['Actual_Churn']=='No Churn')).sum()
FN = ((df['Predicted_Churn']=='No Churn') & (df['Actual_Churn']=='Churn')).sum()
TN = ((df['Predicted_Churn']=='No Churn') & (df['Actual_Churn']=='No Churn')).sum()

accuracy  = (TP + TN) / len(df)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

print(f"=== Model evaluation (threshold: {CHURN_FLAGS}+ red flags → Churn) ===")
print(f"Accuracy : {accuracy:.0%}  ({TP+TN}/{len(df)} correctly classified)")
print(f"Precision: {precision:.0%}  (of predicted churns, how many actually churned)")
print(f"Recall   : {recall:.0%}  (of actual churns, how many were caught)")
print(f"\nConfusion matrix:")
print(f"                 Predicted Churn  Predicted No Churn")
print(f"Actual Churn     {TP:>15}  {FN:>18}")
print(f"Actual No Churn  {FP:>15}  {TN:>18}")

print("\n=== Flag frequency across all customers (% triggered) ===")
print((df[FLAG_COLS].mean() * 100).round(1).sort_values(ascending=False))

print("\n=== Customers the model got WRONG ===")
display(df[df['Correct']==False][['Customer_ID','Company_Name','Outcome','Red_Flag_Count','Predicted_Churn']].reset_index(drop=True))

# ── Step 4: Full output ───────────────────────────────────────────────────────
KEY  = ['Customer_ID', 'Company_Name', 'Industry', 'ACV_USD']
CALC = ['Red_Flag_Count', 'Predicted_Churn', 'Correct'] + FLAG_COLS
RAW  = ['Seats_Licensed', 'Seats_Active_30d', 'Logins_Total_30d',
        'Features_Adopted', 'Last_QBR_Days_Ago', 'Support_Tickets_Open',
        'NPS_Score', 'Payment_Overdue_Days', 'Training_Completion_pct', 'Outcome']
print("\n=== All customers with calculated columns ===")
display(df[KEY + CALC + RAW].reset_index(drop=True))

=== Model evaluation (threshold: 4+ red flags → Churn) ===
Accuracy : 98%  (488/500 correctly classified)
Precision: 100%  (of predicted churns, how many actually churned)
Recall   : 93%  (of actual churns, how many were caught)

Confusion matrix:
                 Predicted Churn  Predicted No Churn
Actual Churn                 163                  12
Actual No Churn                0                 325

=== Flag frequency across all customers (% triggered) ===
flag_qbr         47.8
flag_features    36.6
flag_nps         26.0
flag_training    25.6
flag_tickets     21.2
flag_seat_util   20.0
flag_logins      19.0
flag_payment     15.2
dtype: float64

=== Customers the model got WRONG ===


,Customer_ID,Company_Name,Outcome,Red_Flag_Count,Predicted_Churn
0,O118,Peak Innovations AB,Churned,3,No Churn
1,O146,Cloud Metrics LLC,Churned,3,No Churn
2,O147,Bright Works Inc,Churned,3,No Churn
3,O152,Smart Systems Ltd,Churned,3,No Churn
4,O175,Verve Services SA,Churned,2,No Churn
5,O181,Smart Metrics Inc,Churned,3,No Churn
6,O272,Pinnacle Analytics Ltd,Churned,2,No Churn
7,O342,Verve Holdings GmbH,Churned,3,No Churn
8,O353,Cloud Capital Ltd,Churned,3,No Churn
9,O413,Zenith Services SA,Churned,3,No Churn



=== All customers with calculated columns ===


,Customer_ID,Company_Name,Industry,ACV_USD,Red_Flag_Count,Predicted_Churn,Correct,flag_seat_util,flag_logins,flag_features,...,Seats_Licensed,Seats_Active_30d,Logins_Total_30d,Features_Adopted,Last_QBR_Days_Ago,Support_Tickets_Open,NPS_Score,Payment_Overdue_Days,Training_Completion_pct,Outcome
0,O001,Spark Works SA,Retail,48000,0,No Churn,True,0,0,0,...,70,43,721,3,71,2,55,0,56,Renewed
1,O002,Dynamic Synergies GmbH,Energy,55000,0,No Churn,True,0,0,0,...,69,66,2194,4,43,0,39,0,80,Expanded
2,O003,Catalyst Works AB,Consulting,16000,7,Churn,True,1,1,1,...,23,5,17,1,252,4,-66,0,21,Churned
3,O004,Fusion Holdings AB,Technology,51000,0,No Churn,True,0,0,0,...,76,63,3523,4,11,1,54,0,82,Expanded
4,O005,Titan Works Ltd,Retail,66000,0,No Churn,True,0,0,0,...,86,70,2160,5,21,1,40,0,86,Expanded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,O496,Fusion Innovations Inc,Construction,64000,0,No Churn,True,0,0,0,...,100,59,708,3,88,0,5,0,48,Renewed
496,O497,Cloud Capital Inc,MarTech,11000,6,Churn,True,1,0,1,...,16,4,27,1,355,5,-55,0,22,Churned
497,O498,Pinnacle Group AB,MarTech,78000,0,No Churn,True,0,0,0,...,139,138,8213,5,46,0,37,0,77,Expanded
498,O499,Smart Innovations Corp,Media,65000,0,No Churn,True,0,0,0,...,86,37,864,3,48,1,7,0,59,Renewed


---
## Option C - Signal Ranking by Churn Separation

This approach answers a different question: **which signals actually matter?** Instead of assigning weights up front, we let the data tell us which signals best distinguish customers who churned from those who stayed.

---

### Step 1 - Compare averages between churned and healthy customers

For each signal, we split the dataset into two groups - customers who **churned** and customers who **did not** - and calculate the average (mean) value of that signal in each group.

For example, if churned customers had an average seat utilisation of 22% while healthy customers averaged 81%, that is a large gap. Seat utilisation is clearly connected to churn.

---

### Step 2 - Normalise the gap so signals can be compared fairly

The gap between group averages is not enough on its own. A gap of 10 means something very different for a signal measured in days (Last QBR) versus a percentage (Training Completion).

To make signals comparable, we divide the gap by the **standard deviation** - a measure of how spread out the values are across all customers. Think of it as the "typical" amount of variation in that signal.

```
Normalised gap = (mean_churned − mean_not_churned) / standard_deviation
```

This gives a gap measured in "standard units" rather than the original units of the signal. A normalised gap of 1.5 means churned customers sit 1.5 standard deviations away from the healthy average - a strong and meaningful difference.

---

### Step 3 - Rank signals by absolute gap

We sort by the absolute value of the gap (ignoring direction). A signal can indicate churn either by being **too low** (e.g. seat utilisation, logins - negative gap) or **too high** (e.g. open tickets, QBR days - positive gap). Both are equally useful predictors.

---

### Step 4 - Churn direction score: where does each customer sit?

Using the group means from Step 1, we score each customer on each signal:

```
churn_direction = (customer_value − mean_not_churned) × gap / std_dev
```

- **Positive** → the customer leans toward the churned pattern on this signal
- **Negative** → the customer looks like a healthy (non-churned) account
- **Zero** → the customer sits exactly at the healthy group average

Crucially, each signal is **weighted by its own gap** — signals with a large normalised gap contribute more to the final score than signals with a small gap. This is what makes the ranking meaningful: the score is not just a count of "bad" signals, it is a weighted sum where the data decides the weights.

We sum all signal scores into a single `Churn_Risk_Score` — the higher the score, the more the customer resembles churned accounts across all signals, weighted by how strongly each signal predicts churn.

> **Note:** this section reads the `Outcome` column to compute group means. The column uses white text in the source file — it will load correctly here since pandas ignores font colour.

In [ ]:
#@title Option C - Signal Ranking by Churn Separation
from google.colab import files
pd.options.display.float_format = '{:.1f}'.format

# ── Step 1: Compute group means and rank signals (training step) ──────────────
# gap = (mean_churned − mean_not_churned) / overall_std
df['seat_util_rate']  = df['Seats_Active_30d'] / df['Seats_Licensed'].replace(0, 1)
df['logins_per_seat'] = df['Logins_Total_30d']  / df['Seats_Active_30d'].replace(0, 1)

SIGNALS = [
    'seat_util_rate', 'logins_per_seat', 'Features_Adopted',
    'Last_QBR_Days_Ago', 'Support_Tickets_Open', 'NPS_Score',
    'Payment_Overdue_Days', 'Training_Completion_pct',
]

is_churned = df['Outcome'] == 'Churned'
stats, rank_rows = {}, []
for sig in SIGNALS:
    std  = df[sig].std()
    mc   = df.loc[is_churned,  sig].mean()
    mn   = df.loc[~is_churned, sig].mean()
    gap  = (mc - mn) / std if std else 0
    stats[sig] = {'mean_not_churned': mn, 'std': std, 'gap': gap}
    rank_rows.append({'Signal': sig, 'Mean_Churned': round(mc, 1),
                      'Mean_Not_Churned': round(mn, 1), 'Normalised_Gap': round(gap, 1)})

rank_df = (pd.DataFrame(rank_rows)
             .assign(Abs_Gap=lambda x: x['Normalised_Gap'].abs())
             .sort_values('Abs_Gap', ascending=False)
             .drop(columns='Abs_Gap').reset_index(drop=True))
rank_df.index += 1
print("=== Signal ranking — which signals best predict churn? ===")
display(rank_df)

# ── Step 2: Score each customer using the learned group means ─────────────────
# churn_dir = (value − mean_not_churned) × gap / std
# Weighting by gap (not just its sign) means strongly predictive signals
# contribute proportionally more — direction and magnitude both matter.
# Positive → customer leans toward churned pattern; negative → looks healthy
for sig in SIGNALS:
    s = stats[sig]
    df[f'churn_dir_{sig}'] = (df[sig] - s['mean_not_churned']) * s['gap'] / s['std']

dir_cols = [f'churn_dir_{s}' for s in SIGNALS]
df['Churn_Risk_Score'] = df[dir_cols].sum(axis=1).round(2)

print("\n=== Score distribution by outcome ===")
print(df.groupby('Outcome')['Churn_Risk_Score'].agg(['min', 'mean', 'max']).round(2))

# ── Step 3: Find the threshold that maximises accuracy ───────────────────────
# Scan every unique score value; pick the cutpoint with highest accuracy.
# This is a training-set calibration — the same data used to learn the gaps
# is used to set the decision boundary.

best_acc, best_thresh = 0, 0.0
for t in np.sort(df['Churn_Risk_Score'].unique()):
    acc = ((df['Churn_Risk_Score'] >= t) == is_churned).mean()
    if acc > best_acc:
        best_acc, best_thresh = acc, t

CHURN_THRESHOLD = best_thresh
print(f"\nAuto-selected threshold: {CHURN_THRESHOLD:.2f}  (best training accuracy: {best_acc:.0%})")

# ── Step 4: Predict and evaluate ─────────────────────────────────────────────
df['Predicted_Churn'] = (df['Churn_Risk_Score'] >= CHURN_THRESHOLD).map({True: 'Churn', False: 'No Churn'})
df['Actual_Churn']    = (df['Outcome'] == 'Churned').map({True: 'Churn', False: 'No Churn'})
df['Correct']         = (df['Predicted_Churn'] == df['Actual_Churn'])

TP = ((df['Predicted_Churn']=='Churn') & (df['Actual_Churn']=='Churn')).sum()
FP = ((df['Predicted_Churn']=='Churn') & (df['Actual_Churn']=='No Churn')).sum()
FN = ((df['Predicted_Churn']=='No Churn') & (df['Actual_Churn']=='Churn')).sum()
TN = ((df['Predicted_Churn']=='No Churn') & (df['Actual_Churn']=='No Churn')).sum()

accuracy  = (TP + TN) / len(df)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

print(f"\n=== Model evaluation (threshold: Churn_Risk_Score >= {CHURN_THRESHOLD:.2f}) ===")
print(f"Accuracy : {accuracy:.0%}  ({TP+TN}/{len(df)} correctly classified)")
print(f"Precision: {precision:.0%}  (of predicted churns, how many actually churned)")
print(f"Recall   : {recall:.0%}  (of actual churns, how many were caught)")
print(f"\nConfusion matrix:")
print(f"                 Predicted Churn  Predicted No Churn")
print(f"Actual Churn     {TP:>15}  {FN:>18}")
print(f"Actual No Churn  {FP:>15}  {TN:>18}")

print("\n=== Customers the model got WRONG ===")
display(df[df['Correct']==False][['Customer_ID','Company_Name','Outcome','Churn_Risk_Score','Predicted_Churn']].reset_index(drop=True))

# ── Step 5: Full output ───────────────────────────────────────────────────────
KEY  = ['Customer_ID', 'Company_Name', 'Industry', 'ACV_USD']
CALC = ['Churn_Risk_Score', 'Predicted_Churn', 'Correct',
        'seat_util_rate', 'logins_per_seat'] + dir_cols
RAW  = ['Seats_Licensed', 'Seats_Active_30d', 'Logins_Total_30d',
        'Features_Adopted', 'Last_QBR_Days_Ago', 'Support_Tickets_Open',
        'NPS_Score', 'Payment_Overdue_Days', 'Training_Completion_pct', 'Outcome']
print("\n=== All customers with calculated columns ===")
display(df[KEY + CALC + RAW].reset_index(drop=True))

=== Signal ranking — which signals best predict churn? ===


,Signal,Mean_Churned,Mean_Not_Churned,Normalised_Gap
1,Last_QBR_Days_Ago,261.0,55.0,1.8
2,seat_util_rate,0.2,0.7,-1.7
3,Training_Completion_pct,30.3,74.4,-1.7
4,NPS_Score,-22.4,44.9,-1.7
5,Features_Adopted,1.6,3.7,-1.6
6,logins_per_seat,4.6,28.6,-1.5
7,Support_Tickets_Open,3.3,0.6,1.5
8,Payment_Overdue_Days,12.5,2.0,0.7



=== Score distribution by outcome ===
          min  mean  max
Outcome                 
Churned   7.9  19.1 27.2
Expanded -9.4  -5.4 -1.0
Renewed   0.1   4.6  8.5

Auto-selected threshold: 9.14  (best training accuracy: 100%)

=== Model evaluation (threshold: Churn_Risk_Score >= 9.14) ===
Accuracy : 100%  (499/500 correctly classified)
Precision: 100%  (of predicted churns, how many actually churned)
Recall   : 99%  (of actual churns, how many were caught)

Confusion matrix:
                 Predicted Churn  Predicted No Churn
Actual Churn                 174                   1
Actual No Churn                0                 325

=== Customers the model got WRONG ===


,Customer_ID,Company_Name,Outcome,Churn_Risk_Score,Predicted_Churn
0,O158,Swift Labs Corp,Churned,7.9,No Churn



=== All customers with calculated columns ===


,Customer_ID,Company_Name,Industry,ACV_USD,Churn_Risk_Score,Predicted_Churn,Correct,seat_util_rate,logins_per_seat,churn_dir_seat_util_rate,...,Seats_Licensed,Seats_Active_30d,Logins_Total_30d,Features_Adopted,Last_QBR_Days_Ago,Support_Tickets_Open,NPS_Score,Payment_Overdue_Days,Training_Completion_pct,Outcome
0,O001,Spark Works SA,Retail,48000,4.6,No Churn,True,0.6,16.8,0.6,...,70,43,721,3,71,2,55,0,56,Renewed
1,O002,Dynamic Synergies GmbH,Energy,55000,-3.1,No Churn,True,1.0,33.2,-1.4,...,69,66,2194,4,43,0,39,0,80,Expanded
2,O003,Catalyst Works AB,Consulting,16000,22.5,Churn,True,0.2,3.4,2.9,...,23,5,17,1,252,4,-66,0,21,Churned
3,O004,Fusion Holdings AB,Technology,51000,-4.8,No Churn,True,0.8,55.9,-0.7,...,76,63,3523,4,11,1,54,0,82,Expanded
4,O005,Titan Works Ltd,Retail,66000,-3.2,No Churn,True,0.8,30.9,-0.6,...,86,70,2160,5,21,1,40,0,86,Expanded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,O496,Fusion Innovations Inc,Construction,64000,6.4,No Churn,True,0.6,12.0,0.7,...,100,59,708,3,88,0,5,0,48,Renewed
496,O497,Cloud Capital Inc,MarTech,11000,23.9,Churn,True,0.2,6.8,2.7,...,16,4,27,1,355,5,-55,0,22,Churned
497,O498,Pinnacle Group AB,MarTech,78000,-6.6,No Churn,True,1.0,59.5,-1.6,...,139,138,8213,5,46,0,37,0,77,Expanded
498,O499,Smart Innovations Corp,Media,65000,5.7,No Churn,True,0.4,23.4,1.6,...,86,37,864,3,48,1,7,0,59,Renewed


---
## Which Option Should You Use?

| | Option A - Weighted Score | Option B - Red Flag Counter | Option C - Signal Ranking |
|---|---|---|---|
| **Best for** | CSM dashboards, exec reporting, QBR prep | Daily triage, early alerts, onboarding checks | Calibrating Option A weights, annual strategy review |
| **Data needed** | Current snapshot — no outcome labels needed | Current snapshot — no outcome labels needed | Historical outcomes (Churned / Renewed labels) |
| **Key risk** | Weights are subjective without calibration | All flags carry equal weight | Correlation ≠ causation; threshold must be tuned on training data |
| **Output** | `Health_Score`, `Predicted_Churn` | `Red_Flag_Count`, `Predicted_Churn` | `churn_dir_*`, `Churn_Risk_Score` + Signal Ranking sheet |

In practice, teams may use all three together:
- **Option C** to discover which signals matter most,
- **Option A** to operationalise a composite score, and
- **Option B** to give CSMs a simple daily checklist.

---
*Course: Enterprise Architecture for SaaS Business Models © 2026 Peter Datsichin*